# Road Traffic Sign + Pedestrian Video Assistant

Kaggle-runnable notebook for detecting pedestrians, estimating whether they are on the road, detecting/classifying traffic signs, and overlaying driver-facing instructions on a video.

Recommended Kaggle inputs:
- A video file, for example `/kaggle/input/my-road-video/video.mp4`.
- GTSRB dataset for optional sign-classifier training.
- Optional custom YOLO traffic-sign detector weights or a YOLO-format traffic-sign dataset.

The pedestrian safety logic uses a Cityscapes-trained semantic segmentation model to identify road pixels. A pedestrian alert is shown only when the pedestrian base overlaps the road mask and lies in the driving corridor.

## 1. Install / Import Dependencies

On Kaggle, enable Internet if you want the notebook to download `ultralytics` and the Cityscapes-trained SegFormer model. If Internet is disabled, upload the required model weights as Kaggle datasets and set the paths in the configuration cell.

In [ ]:
import importlib.util
import subprocess
import sys

def ensure_packages(packages):
    missing = []
    for import_name, pip_name in packages:
        if importlib.util.find_spec(import_name) is None:
            missing.append(pip_name)
    if missing:
        print('Installing:', missing)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)

ensure_packages([
    ('ultralytics', 'ultralytics'),
    ('transformers', 'transformers'),
    ('accelerate', 'accelerate'),
])

In [ ]:
from pathlib import Path
import csv
import math
import os
import random
import re
import time
from collections import defaultdict, deque

import cv2
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from ultralytics import YOLO
from transformers import AutoImageProcessor, SegformerForSemanticSegmentation

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

## 2. Configuration

Edit these paths after adding Kaggle datasets from the right sidebar. Output is written to `/kaggle/working`.

In [ ]:
def first_match(root, patterns):
    root = Path(root)
    if not root.exists():
        return None
    for pattern in patterns:
        matches = sorted(root.rglob(pattern))
        if matches:
            return matches[0]
    return None

# Required: auto-detect the first imported Kaggle video, or set this manually.
VIDEO_PATH = first_match('/kaggle/input', ['*.mp4', '*.mov', '*.avi', '*.mkv']) or Path('/kaggle/input/YOUR_VIDEO_DATASET/video.mp4')
OUTPUT_VIDEO_PATH = Path('/kaggle/working/driver_assistant_output.mp4')

# Optional GTSRB dataset root. Common Kaggle layouts include Train/0/*.png or a Train.csv with Path/ClassId.
_gtsrb_csv = first_match('/kaggle/input', ['Train.csv'])
_gtsrb_train_dir = first_match('/kaggle/input', ['Train'])
GTSRB_ROOT = (_gtsrb_csv.parent if _gtsrb_csv else (_gtsrb_train_dir.parent if _gtsrb_train_dir else Path('/kaggle/input/YOUR_GTSRB_DATASET')))
GTSRB_CLASSIFIER_PATH = Path('/kaggle/working/gtsrb_classifier.pt')

# Cityscapes root is used for road/sidewalk context and for negative crops while improving the sign classifier.
CITYSCAPES_ROOT = Path('/kaggle/input/datasets/arjitdsce/cityscapes/cityscapes_data')
if not CITYSCAPES_ROOT.exists():
    _cityscapes_data = first_match('/kaggle/input', ['cityscapes_data'])
    CITYSCAPES_ROOT = _cityscapes_data if _cityscapes_data else CITYSCAPES_ROOT

# Optional custom YOLO traffic-sign detector weights. Use this if you train/upload a detector for sign boxes.
# Without this, the notebook still detects pedestrians and COCO stop signs, and can classify color-based sign proposals if a GTSRB classifier exists.
CUSTOM_TRAFFIC_SIGN_WEIGHTS = None  # e.g. '/kaggle/input/my-sign-yolo/best.pt'

# Cityscapes-trained segmentation model. If Internet is disabled, upload a local model folder and set this path.
SEGFORMER_MODEL_NAME_OR_PATH = 'nvidia/segformer-b0-finetuned-cityscapes-1024-1024'

# YOLO model for pedestrian and stop-sign detection. yolov8n is fast; use yolov8s.pt for stronger accuracy.
YOLO_MODEL = 'yolov8n.pt'

# Runtime controls.
FRAME_STRIDE = 1              # Increase to 2 or 3 for faster processing.
CONFIDENCE = 0.35
ALERT_CONFIRM_FRAMES = 3      # Message must be present for this many processed frames.
MAX_VIDEO_SECONDS = None      # Set to a number for a quick test, or None for full video.

# Traffic-sign fallback controls. Keep color proposals off for clean output; use a YOLO sign detector for real sign boxes.
USE_COLOR_SIGN_PROPOSALS = True
SIGN_ANNOTATION_MIN_PROB = 0.75
SIGN_ANNOTATION_MIN_MARGIN = 0.08
SIGN_ALERT_MIN_PROB = 0.95
SIGN_ALERT_MIN_MARGIN = 0.45
SIGN_PROPOSAL_MAX_BOTTOM_RATIO = 0.85
SIGN_PROPOSAL_MIN_AREA_RATIO = 0.00008
SIGN_PROPOSAL_MAX_AREA_RATIO = 0.045
DEBUG_SIGN_PROPOSALS = False

# Pedestrian danger tuning.
ROAD_OVERLAP_THRESHOLD = 0.15
SIDEWALK_OVERLAP_SAFE_THRESHOLD = 0.20
PED_ALERT_REQUIRE_DRIVING_CORRIDOR = False
MIN_PERSON_HEIGHT_RATIO = 0.07
DRIVING_CORRIDOR_BOTTOM_WIDTH = 0.78
DRIVING_CORRIDOR_TOP_WIDTH = 0.18
DRIVING_CORRIDOR_HORIZON = 0.45

## 3. Traffic Sign Labels and Driver Messages

GTSRB has 43 sign classes. The improved classifier adds class 43 as `not_traffic_sign` to reject background crops.

In [ ]:
GTSRB_LABELS = {
    0: 'speed_limit_20', 1: 'speed_limit_30', 2: 'speed_limit_50', 3: 'speed_limit_60',
    4: 'speed_limit_70', 5: 'speed_limit_80', 6: 'end_speed_limit_80', 7: 'speed_limit_100',
    8: 'speed_limit_120', 9: 'no_passing', 10: 'no_passing_trucks', 11: 'priority_road',
    12: 'priority_next_intersection', 13: 'yield', 14: 'stop', 15: 'no_vehicles',
    16: 'trucks_prohibited', 17: 'no_entry', 18: 'general_danger', 19: 'curve_left',
    20: 'curve_right', 21: 'double_curve', 22: 'bumpy_road', 23: 'slippery_road',
    24: 'road_narrows_right', 25: 'road_work', 26: 'traffic_signals', 27: 'pedestrians_crossing',
    28: 'children_crossing', 29: 'bicycles_crossing', 30: 'ice_snow', 31: 'animal_crossing',
    32: 'end_all_restrictions', 33: 'turn_right_ahead', 34: 'turn_left_ahead', 35: 'ahead_only',
    36: 'go_straight_or_right', 37: 'go_straight_or_left', 38: 'keep_right', 39: 'keep_left',
    40: 'roundabout', 41: 'end_no_passing', 42: 'end_no_passing_trucks',
    43: 'not_traffic_sign'
}

def message_for_sign(label, urgency='normal'):
    label = str(label).lower().replace(' ', '_')
    if label == 'not_traffic_sign':
        return None
    speed = re.search(r'speed[_-]?limit[_-]?(\d+)', label)
    if speed:
        return f"Speed limit is {speed.group(1)} km/h. Please adjust your speed."
    if label in {'stop', 'stop_sign'}:
        return 'Stop sign ahead. Prepare to halt.' if urgency != 'immediate' else 'STOP sign close. Brake and prepare to halt.'
    if label in {'yield', 'give_way'}:
        return 'Yield ahead. Slow down and give way.'
    if label == 'no_entry':
        return 'Wrong way / no entry. Do not enter.'
    if label in {'general_danger', 'danger', 'warning'}:
        return 'Potential risks ahead. Please pay close attention.'
    if label == 'animal_crossing':
        return 'Animal crossing zone. Watch the road edges.'
    if label in {'pedestrians_crossing', 'pedestrian_crossing'}:
        return 'Pedestrian crossing ahead. Slow down and scan the road.'
    if label == 'children_crossing':
        return 'Children crossing area. Slow down and pay close attention.'
    if label == 'road_work':
        return 'Road works ahead. Reduce speed and watch for workers.'
    if label == 'traffic_signals':
        return 'Traffic signals ahead. Be ready to stop.'
    if label == 'slippery_road':
        return 'Slippery road warning. Reduce speed and avoid sudden braking.'
    if label in {'bumpy_road', 'road_narrows_right', 'curve_left', 'curve_right', 'double_curve'}:
        return 'Road hazard ahead. Reduce speed and pay close attention.'
    if label == 'bicycles_crossing':
        return 'Bicycle crossing ahead. Slow down and watch both sides.'
    if label in {'no_passing', 'no_passing_trucks'}:
        return 'No passing zone. Stay in lane.'
    if label in {'keep_right', 'keep_left', 'ahead_only', 'turn_right_ahead', 'turn_left_ahead', 'roundabout'}:
        return f"Mandatory direction sign: {label.replace('_', ' ')}. Follow lane guidance."
    return f"Traffic sign detected: {label.replace('_', ' ')}. Please pay attention."

In [ ]:
NON_SIGN_LABELS = {
    'person', 'pedestrian', 'car', 'cars', 'vehicle', 'vehicles', 'bus', 'truck', 'van', 'train',
    'motorcycle', 'bicycle', 'bike', 'traffic light', 'window', 'door', 'building', 'advertisement',
    'billboard', 'logo', 'license plate', 'plate'
}

def is_probable_sign_label(label):
    normalized = str(label).lower().strip().replace('-', ' ').replace('_', ' ')
    return normalized not in NON_SIGN_LABELS

## 4. Optional: Train a GTSRB Sign Classifier

This classifier is used on sign crops. For best production results, combine it with a custom YOLO sign detector trained on traffic-sign bounding boxes. GTSRB itself is mainly a classification dataset, not a detection dataset.

In [ ]:
class GTSRBDataset(Dataset):
    def __init__(self, root, split='Train', image_size=64, train=True):
        self.root = Path(root)
        self.items = self._discover_items(split)
        if train:
            self.tf = transforms.Compose([
                transforms.Resize((image_size + 8, image_size + 8)),
                transforms.RandomCrop((image_size, image_size)),
                transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.20, hue=0.03),
                transforms.RandomRotation(10),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.3403, 0.3121, 0.3214], std=[0.2724, 0.2608, 0.2669]),
            ])
        else:
            self.tf = transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.3403, 0.3121, 0.3214], std=[0.2724, 0.2608, 0.2669]),
            ])

    def _discover_items(self, split):
        rows = []
        csv_candidates = list(self.root.rglob(f'{split}.csv')) + list(self.root.rglob('Train.csv'))
        for csv_path in csv_candidates[:1]:
            with open(csv_path, newline='') as f:
                for row in csv.DictReader(f):
                    if 'Path' in row and 'ClassId' in row:
                        img_path = (csv_path.parent / row['Path']).resolve()
                        if img_path.exists():
                            rows.append((img_path, int(row['ClassId'])))
            if rows:
                return rows

        split_dirs = [self.root / split, self.root / split.lower(), self.root]
        exts = {'.png', '.jpg', '.jpeg', '.ppm'}
        for base in split_dirs:
            if not base.exists():
                continue
            for class_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
                if class_dir.name.isdigit():
                    class_id = int(class_dir.name)
                    for img_path in class_dir.rglob('*'):
                        if img_path.suffix.lower() in exts:
                            rows.append((img_path, class_id))
            if rows:
                return rows
        return rows

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, label = self.items[idx]
        img = Image.open(path).convert('RGB')
        return self.tf(img), label

def resolve_cityscapes_root():
    configured = globals().get('CITYSCAPES_ROOT', None)
    candidates = []
    if configured is not None:
        candidates.append(Path(configured))
    candidates.extend([
        Path('/kaggle/input/datasets/arjitdsce/cityscapes/cityscapes_data'),
        Path('/kaggle/input/cityscapes/cityscapes_data'),
        Path('/kaggle/input/cityscapes_data'),
    ])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    input_root = Path('/kaggle/input')
    if input_root.exists():
        matches = sorted(input_root.rglob('cityscapes_data'))
        if matches:
            return matches[0]
    return candidates[0]

class NegativeCropDataset(Dataset):
    def __init__(self, cityscapes_root=None, count=12000, image_size=64, train=True):
        self.root = Path(cityscapes_root) if cityscapes_root is not None else resolve_cityscapes_root()
        self.count = count
        self.image_size = image_size
        self.label = 43
        self.paths = []
        for pattern in ('*.png', '*.jpg', '*.jpeg'):
            self.paths.extend(self.root.rglob(pattern))
        # Avoid accidentally using mask-like files if a dataset contains them.
        self.paths = [p for p in self.paths if 'label' not in p.name.lower() and 'mask' not in p.name.lower()]
        if not self.paths:
            print('Warning: no Cityscapes/background images found. Negative class will be empty.')
        self.tf = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.20, hue=0.03) if train else transforms.Lambda(lambda x: x),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.3403, 0.3121, 0.3214], std=[0.2724, 0.2608, 0.2669]),
        ])

    def __len__(self):
        return self.count if self.paths else 0

    def __getitem__(self, idx):
        path = random.choice(self.paths)
        img = Image.open(path).convert('RGB')
        w, h = img.size
        crop_size = random.randint(max(32, min(w, h) // 12), max(40, min(w, h) // 3))
        crop_size = min(crop_size, w, h)
        x1 = random.randint(0, max(0, w - crop_size))
        y1 = random.randint(0, max(0, h - crop_size))
        crop = img.crop((x1, y1, x1 + crop_size, y1 + crop_size))
        return self.tf(crop), self.label

class SmallGTSRBCNN(nn.Module):
    def __init__(self, num_classes=44):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
            nn.Flatten(), nn.Dropout(0.35), nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.net(x)

SIGN_TF = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.3403, 0.3121, 0.3214], std=[0.2724, 0.2608, 0.2669]),
])

def make_classifier_loaders(root=GTSRB_ROOT, val_fraction=0.15, batch_size=128, negative_count=12000):
    sign_dataset = GTSRBDataset(root, split='Train', train=True)
    if len(sign_dataset) == 0:
        raise FileNotFoundError(f'No GTSRB images found under {root}. Import the dataset and update GTSRB_ROOT.')
    val_count = max(1, int(len(sign_dataset) * val_fraction))
    train_count = len(sign_dataset) - val_count
    generator = torch.Generator().manual_seed(42)
    train_signs, val_signs = torch.utils.data.random_split(sign_dataset, [train_count, val_count], generator=generator)

    # Validation should be deterministic, so use a non-augmented copy for the validation indices.
    val_base = GTSRBDataset(root, split='Train', train=False)
    val_signs = torch.utils.data.Subset(val_base, val_signs.indices)

    cityscapes_root = resolve_cityscapes_root()
    print('Using Cityscapes/background root for negative crops:', cityscapes_root)
    neg_train = NegativeCropDataset(cityscapes_root, count=negative_count, train=True)
    neg_val = NegativeCropDataset(cityscapes_root, count=max(1000, negative_count // 5), train=False)
    train_dataset = torch.utils.data.ConcatDataset([train_signs, neg_train])
    val_dataset = torch.utils.data.ConcatDataset([val_signs, neg_val])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    print(f'Training samples: {len(train_dataset)} | validation samples: {len(val_dataset)} | classes: 44 including not_traffic_sign')
    return train_loader, val_loader

@torch.no_grad()
def evaluate_classifier(model, loader):
    model.eval()
    correct, total = 0, 0
    negative_correct, negative_total = 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total += y.numel()
        neg_mask = y == 43
        if neg_mask.any():
            negative_correct += (pred[neg_mask] == y[neg_mask]).sum().item()
            negative_total += neg_mask.sum().item()
    return correct / max(1, total), negative_correct / max(1, negative_total)

def train_gtsrb_classifier(root=GTSRB_ROOT, output_path=GTSRB_CLASSIFIER_PATH, epochs=20, batch_size=128, negative_count=12000):
    train_loader, val_loader = make_classifier_loaders(root, batch_size=batch_size, negative_count=negative_count)
    model = SmallGTSRBCNN(num_classes=44).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))
    best_val = 0.0
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
                logits = model(x)
                loss = loss_fn(logits, y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            total += x.size(0)
        scheduler.step()
        val_acc, neg_acc = evaluate_classifier(model, val_loader)
        train_acc = correct / max(1, total)
        print(f'Epoch {epoch + 1}/{epochs} - loss={total_loss / total:.4f} train_acc={train_acc:.3f} val_acc={val_acc:.3f} neg_acc={neg_acc:.3f}')
        if val_acc > best_val:
            best_val = val_acc
            torch.save({'model_state': model.state_dict(), 'num_classes': 44, 'labels': GTSRB_LABELS}, output_path)
            print(f'  saved best checkpoint to {output_path}')
    model.load_state_dict(torch.load(output_path, map_location=DEVICE)['model_state'])
    model.eval()
    return model

def load_gtsrb_classifier(path=GTSRB_CLASSIFIER_PATH):
    if not Path(path).exists():
        return None
    checkpoint = torch.load(path, map_location=DEVICE)
    if isinstance(checkpoint, dict) and 'model_state' in checkpoint:
        num_classes = int(checkpoint.get('num_classes', 44))
        state = checkpoint['model_state']
    else:
        num_classes = 43
        state = checkpoint
    model = SmallGTSRBCNN(num_classes=num_classes).to(DEVICE)
    model.load_state_dict(state)
    model.eval()
    return model

# Recommended: train once, then comment the training line and only load the saved classifier.
# Use more epochs if validation accuracy is still rising; the negative class is the main false-positive fix.
sign_classifier = train_gtsrb_classifier(epochs=20, negative_count=12000)
# sign_classifier = load_gtsrb_classifier()
print('GTSRB classifier loaded:', sign_classifier is not None)

## 5. Optional: Train a YOLO Traffic-Sign Detector

If you import a YOLO-format sign detection dataset with a `data.yaml`, this trains a detector that can locate many sign types directly. GTSRB alone does not include full-scene bounding boxes, so a detector dataset gives much better sign localization.

In [ ]:
def train_yolo_sign_detector(data_yaml, epochs=30, imgsz=640, base_model='yolov8n.pt'):
    model = YOLO(base_model)
    results = model.train(data=str(data_yaml), epochs=epochs, imgsz=imgsz, device=0 if DEVICE == 'cuda' else 'cpu')
    print('Best weights are usually saved under /kaggle/working/runs/detect/train/weights/best.pt')
    return results

# Example:
# train_yolo_sign_detector('/kaggle/input/my-yolo-traffic-signs/data.yaml', epochs=40)

## 6. Detection, Segmentation, and Spatial Logic

In [ ]:
class RoadSegmenter:
    def __init__(self, model_name_or_path=SEGFORMER_MODEL_NAME_OR_PATH):
        self.processor = AutoImageProcessor.from_pretrained(model_name_or_path)
        self.model = SegformerForSemanticSegmentation.from_pretrained(model_name_or_path).to(DEVICE)
        self.model.eval()
        labels = {int(k): v.lower() for k, v in self.model.config.id2label.items()}
        self.road_ids = [i for i, name in labels.items() if name == 'road']
        self.sidewalk_ids = [i for i, name in labels.items() if name == 'sidewalk']
        if not self.road_ids:
            raise ValueError('The segmentation model does not expose a road class.')

    @torch.no_grad()
    def predict_masks(self, frame_bgr):
        image_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        inputs = self.processor(images=image_rgb, return_tensors='pt').to(DEVICE)
        logits = self.model(**inputs).logits
        logits = torch.nn.functional.interpolate(
            logits,
            size=image_rgb.shape[:2],
            mode='bilinear',
            align_corners=False,
        )
        pred = logits.argmax(dim=1)[0].detach().cpu().numpy().astype(np.uint8)
        road = np.isin(pred, self.road_ids)
        sidewalk = np.isin(pred, self.sidewalk_ids) if self.sidewalk_ids else np.zeros_like(road, dtype=bool)
        return road, sidewalk

def driving_corridor_bounds(y, h, w):
    horizon_y = h * DRIVING_CORRIDOR_HORIZON
    if y <= horizon_y:
        width_ratio = DRIVING_CORRIDOR_TOP_WIDTH
    else:
        t = min(1.0, (y - horizon_y) / max(1.0, h - horizon_y))
        width_ratio = DRIVING_CORRIDOR_TOP_WIDTH + t * (DRIVING_CORRIDOR_BOTTOM_WIDTH - DRIVING_CORRIDOR_TOP_WIDTH)
    half = 0.5 * width_ratio * w
    return int(w / 2 - half), int(w / 2 + half)

def is_pedestrian_dangerous(box, road_mask, sidewalk_mask, frame_shape):
    h, w = frame_shape[:2]
    x1, y1, x2, y2 = [int(v) for v in box]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w - 1, x2), min(h - 1, y2)
    box_h = max(1, y2 - y1)
    if box_h / h < MIN_PERSON_HEIGHT_RATIO:
        return False, 0.0, 0.0

    base_y1 = max(y1, int(y2 - 0.12 * box_h))
    base_x1 = int(x1 + 0.20 * (x2 - x1))
    base_x2 = int(x2 - 0.20 * (x2 - x1))
    road_region = road_mask[base_y1:y2 + 1, base_x1:base_x2 + 1]
    sidewalk_region = sidewalk_mask[base_y1:y2 + 1, base_x1:base_x2 + 1]
    road_overlap = float(road_region.mean()) if road_region.size else 0.0
    sidewalk_overlap = float(sidewalk_region.mean()) if sidewalk_region.size else 0.0

    foot_x = (x1 + x2) / 2
    left, right = driving_corridor_bounds(y2, h, w)
    in_corridor = left <= foot_x <= right
    corridor_ok = True if not PED_ALERT_REQUIRE_DRIVING_CORRIDOR else in_corridor
    clearly_sidewalk = sidewalk_overlap >= SIDEWALK_OVERLAP_SAFE_THRESHOLD and sidewalk_overlap >= road_overlap
    on_street = road_overlap >= ROAD_OVERLAP_THRESHOLD and road_overlap > sidewalk_overlap
    return bool(on_street and not clearly_sidewalk and corridor_ok), road_overlap, sidewalk_overlap

def sign_candidates_by_color(frame_bgr, min_area=None):
    if min_area is None:
        min_area = int(frame_bgr.shape[0] * frame_bgr.shape[1] * SIGN_PROPOSAL_MIN_AREA_RATIO)
    max_area = int(frame_bgr.shape[0] * frame_bgr.shape[1] * SIGN_PROPOSAL_MAX_AREA_RATIO)
    hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)
    red1 = cv2.inRange(hsv, (0, 70, 50), (10, 255, 255))
    red2 = cv2.inRange(hsv, (170, 70, 50), (180, 255, 255))
    blue = cv2.inRange(hsv, (90, 60, 50), (135, 255, 255))
    yellow = cv2.inRange(hsv, (15, 60, 80), (40, 255, 255))
    mask = red1 | red2 | blue | yellow
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    h, w = frame_bgr.shape[:2]
    for c in contours:
        area = cv2.contourArea(c)
        if area < min_area or area > max_area:
            continue
        x, y, bw, bh = cv2.boundingRect(c)
        aspect = bw / max(1, bh)
        bottom_ratio = (y + bh) / max(1, h)
        fill_ratio = area / max(1, bw * bh)
        if 0.45 <= aspect <= 1.85 and 0.08 <= fill_ratio <= 0.92 and bottom_ratio <= SIGN_PROPOSAL_MAX_BOTTOM_RATIO:
            pad = int(0.12 * max(bw, bh))
            boxes.append((max(0, x - pad), max(0, y - pad), min(w - 1, x + bw + pad), min(h - 1, y + bh + pad)))
    return boxes

@torch.no_grad()
def classify_sign_crop(frame_bgr, box, classifier, min_prob=SIGN_ANNOTATION_MIN_PROB, min_margin=SIGN_ANNOTATION_MIN_MARGIN):
    if classifier is None:
        return None, 0.0, 0.0
    x1, y1, x2, y2 = [int(v) for v in box]
    crop = frame_bgr[max(0, y1):max(0, y2), max(0, x1):max(0, x2)]
    if crop.size == 0:
        return None, 0.0, 0.0
    pil = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    x = SIGN_TF(pil).unsqueeze(0).to(DEVICE)
    prob = torch.softmax(classifier(x), dim=1)[0]
    top2 = torch.topk(prob, k=2)
    conf = top2.values[0]
    cls = top2.indices[0]
    margin = float((top2.values[0] - top2.values[1]).item())
    conf = float(conf.item())
    if conf < min_prob or margin < min_margin:
        return None, conf, margin
    label = GTSRB_LABELS.get(int(cls.item()), 'not_traffic_sign')
    if label == 'not_traffic_sign':
        return None, conf, margin
    return label, conf, margin

class AlertFilter:
    def __init__(self, confirm_frames=3):
        self.confirm_frames = confirm_frames
        self.counts = defaultdict(int)

    def update(self, events):
        active = {event['key'] for event in events}
        for key in list(self.counts):
            if key not in active:
                self.counts[key] = 0
        confirmed = []
        for event in events:
            self.counts[event['key']] += 1
            if self.counts[event['key']] >= self.confirm_frames:
                confirmed.append(event)
        return confirmed

## 7. Video Pipeline

In [ ]:
class DriverAssistant:
    def __init__(self):
        self.yolo = YOLO(YOLO_MODEL)
        self.sign_yolo = YOLO(CUSTOM_TRAFFIC_SIGN_WEIGHTS) if CUSTOM_TRAFFIC_SIGN_WEIGHTS else None
        self.segmenter = RoadSegmenter()
        self.alert_filter = AlertFilter(ALERT_CONFIRM_FRAMES)

    def _draw_box(self, frame, box, label, color):
        x1, y1, x2, y2 = [int(v) for v in box]
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, label, (x1, max(18, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)

    def _overlay_messages(self, frame, messages):
        if not messages:
            return
        h, w = frame.shape[:2]
        unique = []
        seen = set()
        for event in sorted(messages, key=lambda e: e.get('priority', 5)):
            if event['message'] not in seen:
                seen.add(event['message'])
                unique.append(event['message'])
        panel_h = min(h - 20, 24 + 34 * len(unique))
        overlay = frame.copy()
        cv2.rectangle(overlay, (12, 12), (w - 12, 12 + panel_h), (0, 0, 0), -1)
        frame[:] = cv2.addWeighted(overlay, 0.55, frame, 0.45, 0)
        for i, msg in enumerate(unique):
            y = 44 + i * 34
            color = (40, 40, 255) if msg.startswith('CRITICAL') or msg.startswith('STOP') else (0, 255, 255)
            cv2.putText(frame, msg[:110], (24, y), cv2.FONT_HERSHEY_SIMPLEX, 0.75, color, 2, cv2.LINE_AA)

    def _detect_people_and_stop_signs(self, frame, road_mask, sidewalk_mask):
        events = []
        result = self.yolo.predict(frame, conf=CONFIDENCE, verbose=False)[0]
        names = result.names
        for box in result.boxes:
            cls_id = int(box.cls.item())
            label = names.get(cls_id, str(cls_id)).lower()
            xyxy = box.xyxy[0].detach().cpu().numpy()
            conf = float(box.conf.item())
            if label == 'person':
                dangerous, road_overlap, sidewalk_overlap = is_pedestrian_dangerous(xyxy, road_mask, sidewalk_mask, frame.shape)
                if dangerous:
                    events.append({
                        'key': 'pedestrian_on_road',
                        'message': 'CRITICAL: Pedestrian on road. Brake and pay close attention!',
                        'priority': 0,
                        'box': xyxy,
                    })
                    self._draw_box(frame, xyxy, f'person on street {conf:.2f} road={road_overlap:.2f}', (0, 0, 255))
                else:
                    self._draw_box(frame, xyxy, f'person no alert {conf:.2f} road={road_overlap:.2f} side={sidewalk_overlap:.2f}', (80, 180, 80))
            elif label == 'stop sign':
                box_h = (xyxy[3] - xyxy[1]) / frame.shape[0]
                urgency = 'immediate' if box_h > 0.10 else 'normal'
                events.append({
                    'key': 'stop_sign',
                    'message': message_for_sign('stop', urgency=urgency),
                    'priority': 1,
                    'box': xyxy,
                })
                self._draw_box(frame, xyxy, f'stop sign {conf:.2f}', (0, 255, 255))
        return events

    def _detect_custom_signs(self, frame):
        events = []
        if self.sign_yolo is not None:
            result = self.sign_yolo.predict(frame, conf=CONFIDENCE, verbose=False)[0]
            names = result.names
            for box in result.boxes:
                cls_id = int(box.cls.item())
                label = names.get(cls_id, str(cls_id))
                if not is_probable_sign_label(label):
                    continue
                xyxy = box.xyxy[0].detach().cpu().numpy()
                conf = float(box.conf.item())
                msg = message_for_sign(label)
                events.append({'key': f'sign_{label}', 'message': msg, 'priority': 2, 'box': xyxy})
                self._draw_box(frame, xyxy, f'{label} {conf:.2f} | {msg[:48]}', (255, 200, 0))
            return events

        # Fallback: color proposals + GTSRB crop classifier. This is useful, but a custom detector is more robust.
        if sign_classifier is None or not USE_COLOR_SIGN_PROPOSALS:
            return events
        candidates = sign_candidates_by_color(frame)
        if DEBUG_SIGN_PROPOSALS:
            for candidate in candidates:
                self._draw_box(frame, candidate, 'sign candidate', (160, 160, 160))
        for xyxy in candidates:
            label, conf, margin = classify_sign_crop(frame, xyxy, sign_classifier)
            if label is None:
                continue
            msg = message_for_sign(label)
            self._draw_box(frame, xyxy, f'{label} {conf:.2f} | {msg[:48]}', (255, 200, 0))
            if conf >= SIGN_ALERT_MIN_PROB and margin >= SIGN_ALERT_MIN_MARGIN:
                events.append({'key': f'sign_{label}', 'message': msg, 'priority': 2, 'box': xyxy})
        return events

    def process_frame(self, frame):
        road_mask, sidewalk_mask = self.segmenter.predict_masks(frame)

        # Road overlay for debugging; comment this out if you want a clean video.
        road_overlay = np.zeros_like(frame)
        road_overlay[road_mask] = (70, 70, 70)
        frame = cv2.addWeighted(frame, 0.88, road_overlay, 0.12, 0)

        events = []
        events.extend(self._detect_people_and_stop_signs(frame, road_mask, sidewalk_mask))
        events.extend(self._detect_custom_signs(frame))
        confirmed = self.alert_filter.update(events)
        self._overlay_messages(frame, confirmed)
        return frame, confirmed

def process_video(video_path=VIDEO_PATH, output_path=OUTPUT_VIDEO_PATH):
    video_path = Path(video_path)
    if not video_path.exists():
        raise FileNotFoundError(f'Video not found: {video_path}. Add a Kaggle video dataset and update VIDEO_PATH.')

    assistant = DriverAssistant()
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    max_frames = total_frames
    if MAX_VIDEO_SECONDS is not None:
        max_frames = min(max_frames, int(MAX_VIDEO_SECONDS * fps))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(output_path), fourcc, fps / FRAME_STRIDE, (width, height))

    start = time.time()
    processed = 0
    frame_idx = 0
    try:
        while cap.isOpened() and frame_idx < max_frames:
            ok, frame = cap.read()
            if not ok:
                break
            if frame_idx % FRAME_STRIDE == 0:
                annotated, alerts = assistant.process_frame(frame)
                writer.write(annotated)
                processed += 1
                if processed % 25 == 0:
                    elapsed = time.time() - start
                    print(f'Processed {processed} frames ({processed / max(1e-6, elapsed):.2f} fps). Active alerts: {[a["message"] for a in alerts]}')
            frame_idx += 1
    finally:
        cap.release()
        writer.release()
    print(f'Done. Wrote: {output_path}')
    return output_path

## 8. Run on Your Video

In [ ]:
# 1. Update VIDEO_PATH in the configuration cell.
# 2. Optionally train/load the GTSRB classifier and/or set CUSTOM_TRAFFIC_SIGN_WEIGHTS.
# 3. Run this cell.

output_path = process_video(VIDEO_PATH, OUTPUT_VIDEO_PATH)
output_path

## 9. Preview Output in Kaggle

In [ ]:
from IPython.display import Video, display

if OUTPUT_VIDEO_PATH.exists():
    display(Video(str(OUTPUT_VIDEO_PATH), embed=True, width=900))
else:
    print('Run the video pipeline first.')

## Notes for Better Accuracy

- Use a custom traffic-sign detector trained on full-road images with bounding boxes. GTSRB is excellent for sign classification, but not enough by itself for high-quality localization.
- Keep `ALERT_CONFIRM_FRAMES` between 3 and 5 to reduce ghost detections.
- For pedestrians, tune `ROAD_OVERLAP_THRESHOLD`, `DRIVING_CORRIDOR_*`, and `MIN_PERSON_HEIGHT_RATIO` on your videos. This avoids warnings for pedestrians safely on the sidewalk.
- Treat this as a research/education prototype, not a safety-certified driving system.